In [36]:
# Dataframe and Plotting libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss


pd.set_option('display.max_columns', None)
# from google.colab import files
import warnings
warnings.filterwarnings('ignore')

In [37]:
# Load the files into a Pandas Dataframe
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [38]:
print("Train shape: "+ str(train.shape))
print("Test shape: "+ str(test.shape))

Train shape: (373, 12)
Test shape: (558, 11)


In [39]:
train.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,MERCHANT_CATEGORIZED_AS,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-05-04 10:25:50.588042+00,UONSDA CHURCH MOGERE MWAYO TITHE,Rent / Mortgage,1700,2022-05-04 13:56:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_04mk78fa
1,2021-10-25 16:18:38.586837+00,PARK N GO,Transport & Fuel,100,2021-10-24 14:12:00+00,False,25.0,Female,4,90000,ID_U9WZMGJZ,ID_04xkfb07
2,2022-05-20 14:17:30.917297+00,SAFARICOM OFFERS TUNUKIWA,Data & WiFi,20,2022-05-27 12:46:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_051urwuw
3,2022-04-12 15:59:14.139347+00,ZILLIONS CREDIT LIMITED,Loan Repayment,3800,2022-02-07 12:16:00+00,False,NaN,Male,5,252000,ID_3JA0MAFB,ID_0e1om7rz
4,2022-05-04 10:24:26.709045+00,MICHAEL MUSEMBI,Bills & Fees,60,2022-05-04 15:28:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0kfcoawb


In [40]:
test.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-06-01 10:25:16.7131+00,KCB PAYBILL AC,150000,2022-05-05 08:29:00+00,True,NaN,Male,5,150000,ID_O8P8YS18,ID_00x9h2yx
1,2022-03-16 13:05:51.851102+00,IPAY LTD,7394,2019-10-05 16:02:00+00,False,NaN,Female,1,10000,ID_40L9OTIM,ID_01db594f
2,2022-03-10 18:25:42.112294+00,JOHN NJIRU GAKUNJU,200,2021-11-25 09:56:00+00,False,NaN,Female,1,85000,ID_1I8XYBWK,ID_07pm99no
3,2022-04-12 15:59:14.139347+00,UONSDA CHURCH OFFERING-MAIN,50,2022-04-11 14:48:00+00,False,NaN,Male,2,10000,ID_GR569FUO,ID_09jryhkd
4,2022-05-04 10:24:26.709045+00,M-SHWARI LOCK SAVINGS,10000,2022-05-04 13:59:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0ccv924c


# Prepare

In [41]:
test.isna().sum()

MERCHANT_CATEGORIZED_AT                    0
MERCHANT_NAME                              0
PURCHASE_VALUE                             0
PURCHASED_AT                               0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY      0
USER_AGE                                 473
USER_GENDER                                5
USER_HOUSEHOLD                             0
USER_INCOME                                0
USER_ID                                    0
Transaction_ID                             0
dtype: int64

In [42]:
features = train[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY","USER_GENDER","USER_INCOME","USER_HOUSEHOLD"]]

In [43]:
target = train["MERCHANT_CATEGORIZED_AS"]

In [44]:
features["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(features["MERCHANT_CATEGORIZED_AT"])
features["MERCHANT_CATEGORIZED_AT"] = features['MERCHANT_CATEGORIZED_AT'].dt.hour

In [45]:
features["PURCHASED_AT"] = pd.to_datetime(features["PURCHASED_AT"])
features["PURCHASED_AT"] = features['PURCHASED_AT'].dt.hour

In [46]:
def binate(x):
    if x == True:
        return 1
    elif x == False:
        return 0
    elif x == "Male":
        return 1
    elif x == "Female":
        return 0
    else:
        return 1

In [47]:
features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [48]:
features["USER_GENDER"] = features["USER_GENDER"].apply(binate)

In [49]:
features.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_GENDER,USER_INCOME,USER_HOUSEHOLD
0,10,1700,13,0,1,10000,3
1,16,100,14,0,0,90000,4
2,14,20,12,0,1,10000,3
3,15,3800,12,0,1,252000,5
4,10,60,15,0,1,10000,3


In [50]:
features.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_GENDER                              0
USER_INCOME                              0
USER_HOUSEHOLD                           0
dtype: int64

# Prepare TEST

In [51]:
tfeatures = test[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY","USER_GENDER","USER_INCOME","USER_HOUSEHOLD"]]

In [52]:
tfeatures["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(tfeatures["MERCHANT_CATEGORIZED_AT"])
tfeatures["MERCHANT_CATEGORIZED_AT"] = tfeatures['MERCHANT_CATEGORIZED_AT'].dt.hour

In [53]:
tfeatures["PURCHASED_AT"] = pd.to_datetime(tfeatures["PURCHASED_AT"])
tfeatures["PURCHASED_AT"] = tfeatures['PURCHASED_AT'].dt.hour

In [54]:
tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [55]:
tfeatures["USER_GENDER"] = tfeatures["USER_GENDER"].apply(binate)

In [56]:
tfeatures.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_GENDER,USER_INCOME,USER_HOUSEHOLD
0,10,150000,8,1,1,150000,5
1,13,7394,16,0,0,10000,1
2,18,200,9,0,0,85000,1
3,15,50,14,0,1,10000,2
4,10,10000,13,0,1,10000,3


In [57]:
tfeatures.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_GENDER                              0
USER_INCOME                              0
USER_HOUSEHOLD                           0
dtype: int64

In [58]:
tfeatures.shape

(558, 7)

# MODEL

In [59]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

In [60]:
svm = LinearSVC()
clf = CalibratedClassifierCV(svm) 
clf.fit(features, target)
predictions = clf.predict_proba(tfeatures)

In [61]:
predictions.shape

(558, 13)

In [62]:
predictions[0:3]

array([[0.14911235, 0.13763819, 0.03153972, 0.04011956, 0.12992889,
        0.10992684, 0.10722886, 0.01796748, 0.01963265, 0.11519911,
        0.01391696, 0.0351449 , 0.09264448],
       [0.20660388, 0.11051647, 0.01382987, 0.04282682, 0.09105514,
        0.10419373, 0.13240546, 0.02352115, 0.01763663, 0.12678068,
        0.01753287, 0.05764129, 0.05545602],
       [0.19870792, 0.11327568, 0.01444253, 0.03682775, 0.09894056,
        0.10632205, 0.13150891, 0.02323432, 0.01944253, 0.10914919,
        0.01613306, 0.05751366, 0.07450185]])

In [63]:
clf.classes_

array(['Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel'], dtype=object)

In [64]:
predictions_df = pd.DataFrame(predictions)
predictions_df.columns = clf.classes_
predictions_df.head()


,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,0.149112,0.137638,0.031540,0.040120,0.129929,0.109927,0.107229,0.017967,0.019633,0.115199,0.013917,0.035145,0.092644
1,0.206604,0.110516,0.013830,0.042827,0.091055,0.104194,0.132405,0.023521,0.017637,0.126781,0.017533,0.057641,0.055456
2,0.198708,0.113276,0.014443,0.036828,0.098941,0.106322,0.131509,0.023234,0.019443,0.109149,0.016133,0.057514,0.074502
3,0.218962,0.107305,0.013105,0.041029,0.087381,0.099809,0.130162,0.023199,0.017112,0.119722,0.017125,0.058638,0.066451
4,0.202662,0.111535,0.014071,0.043388,0.092170,0.105534,0.133010,0.023564,0.017789,0.128873,0.017648,0.057075,0.052685


In [65]:
ss.head()

,Transaction_ID,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,ID_00x9h2yx,0,0,0,0,0,0,0,0,0,0,0,0,0
1,ID_01db594f,0,0,0,0,0,0,0,0,0,0,0,0,0
2,ID_07pm99no,0,0,0,0,0,0,0,0,0,0,0,0,0
3,ID_09jryhkd,0,0,0,0,0,0,0,0,0,0,0,0,0
4,ID_0ccv924c,0,0,0,0,0,0,0,0,0,0,0,0,0


In [66]:
predictions_df['Transaction_ID'] = ss['Transaction_ID']
predictions_df.head()

,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel,Transaction_ID
0,0.149112,0.137638,0.031540,0.040120,0.129929,0.109927,0.107229,0.017967,0.019633,0.115199,0.013917,0.035145,0.092644,ID_00x9h2yx
1,0.206604,0.110516,0.013830,0.042827,0.091055,0.104194,0.132405,0.023521,0.017637,0.126781,0.017533,0.057641,0.055456,ID_01db594f
2,0.198708,0.113276,0.014443,0.036828,0.098941,0.106322,0.131509,0.023234,0.019443,0.109149,0.016133,0.057514,0.074502,ID_07pm99no
3,0.218962,0.107305,0.013105,0.041029,0.087381,0.099809,0.130162,0.023199,0.017112,0.119722,0.017125,0.058638,0.066451,ID_09jryhkd
4,0.202662,0.111535,0.014071,0.043388,0.092170,0.105534,0.133010,0.023564,0.017789,0.128873,0.017648,0.057075,0.052685,ID_0ccv924c


In [67]:
predictions_df = predictions_df[['Transaction_ID', 'Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel']]

In [68]:
# Create the submission csv file
predictions_df.to_csv('jetting_off.csv', index = False)

In [69]:
predictions_df.shape

(558, 14)

In [70]:
clf.score(features,target)

0.1903485254691689